# Raw 데이터 수집

국가법령정보 API에서 부동산/임대차 관련 문서를 수집하여 `data/raw/{target}.jsonl`에 저장한다.

| target  | 설명                  | 중복 제거 기준       |
| ------- | --------------------- | -------------------- |
| `acr`   | 국민권익위원회 결정문 | `결정문일련번호`     |
| `eflaw` | 현행법령              | `법령ID`             |
| `prec`  | 판례                  | `판례일련번호`       |
| `expc`  | 법령해석례            | `법령해석례일련번호` |

> 각 섹션은 독립적으로 실행 가능. 전체 실행 시 **Kernel → Restart & Run All**.


## 0. 공통 설정


In [23]:
import sys

sys.path.insert(0, "..")
from src.api import fetch_list, save_raw, fetch_details

## 3. 판례 (prec)


In [24]:
# 0. 임대차 분쟁에서 자주 다뤄지는 판례 관련 키워드
PREC_QUERIES: list[str] = [
    "주택",     # 주택: 거주와 직결된 임대차 판례들을 조회하기 위함
    "임대차",   # 임대차: 주제의 근간
    "계약"
]

# 1. 계약은 다른 두 단어와 달리 최고의 선택지는 아닌 것처럼 보입니다.
    # 하지만 임대차 계약 상에서의 시시비비를 다투는 판례를 찾는 것이 특약과 관련된 내용을 내포할 것이라 판단하여 계약을 넣었음.
# 2. 계약 대신 특약을 넣어도 됐을 거긴 한데, 차후 서비스가 확장될 가능성을 고려하면 계약이 더 포괄적이라 계약을 선택했음.
    # 필요에 따라 특약으로 넣어도 될 듯 합니다.

In [4]:
# 1. '판례 목록'에서 판례 키워드(PREC_QUERIES)를 내용에 포함하고 있는 판례들만 추출하기

seen_prec: set[str] = set() # 수집된 판례일련번호
prec_items: list[dict] = [] 

items = fetch_list("prec", query=PREC_QUERIES, max_items=None, extra_params={"search": 2})
    # 기존 코드에는 for문으로 했으나, query를 중복해서 주려면(n중쿼리) list를 넣어야해서 for문은 제거하고 query 리스트 전체 입력
for item in items:
    doc_id = str(item.get("판례일련번호", ""))
    # doc_id가 비어있거나 이미 수집한 항목이면 skip
    if doc_id and doc_id not in seen_prec:
        seen_prec.add(doc_id)
        prec_items.append(item)

save_raw(prec_items, target="prec", mode="w")

print(len(prec_items))

1717


In [ ]:
# cf) 삼중쿼리로 들어가는 게 맞는지 확인하는 코드

for q in PREC_QUERIES:
    items = fetch_list("prec", query=q, max_items=None, extra_params={"search": 2})
    print(f"[{q}] → {len(items)}건")

[주택] → 17186건
[임대차] → 7281건
[계약] → 49513건


In [5]:
# 2. 1번 과정에서 추출한 목록에 판례 내용을 추가하기.
prec_details = fetch_details(
  target="prec",
  items=prec_items,
  id_field="판례일련번호",
)

# 목록만 있는 prec.jsonl 덮어쓰기
result = save_raw(prec_details, target="prec", mode="w")
print(len(prec_details))

1717


In [ ]:
# 3. 판례 상세 링크에 value값 노출 >>> 제거하는 코드 짜야..



---

## 5. 수집 결과 요약


In [2]:
from pathlib import Path

path = Path("../data/raw/prec.jsonl")

# JSONL은 한 줄 = 한 건이므로 줄 수 = 수집 건수
if path.exists():
    count = sum(1 for _ in path.open(encoding="utf-8"))
    print(f"prec: {count:5d}건  →  {path}")
else:
    print(f"prec: 파일 없음")

prec:  1717건  →  ..\data\raw\prec.jsonl


---

---

---

# 수집한 RAW 데이터 전처리

- 전처리된 데이터는 data/processed 폴더에 넣어주세요

### 데이터 기본 정보

In [12]:
import pandas as pd
from pathlib import Path

path = Path("../data/raw/prec.jsonl")
df = pd.read_json(path, lines=True)   # lines = True : 한 줄에 JSON 하나

In [13]:
df

,id,사건번호,데이터출처명,사건종류코드,사건종류명,선고,선고일자,판례일련번호,판결유형,법원종류코드,법원명,판례상세링크,사건명,본문
0,1,2025두34604,대법원,400108,세무,선고,2026.01.29,616581,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=616...,증여세부과처분취소,{'PrecService': {'판시사항': '<br/> [1] 구 상속세 및 증여...
1,2,2025가단107133,대법원,400101,민사,선고,2026.01.23,616251,판결 : 확정,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=616...,배당이의,{'PrecService': {'판시사항': '<br/> 甲 소유의 주택에 관하여 ...
2,3,2025다213466,대법원,400101,민사,선고,2026.01.08,615767,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=615...,보증금,{'PrecService': {'판시사항': '<br/> [1] 주택임대차보호법 제...
3,4,수원고등법원-2025-나-12218,국세법령정보시스템,400101,민사,,2025.12.10,615921,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=615...,원고가 이 사건 경매절차에서 배당받을 권리가 있는지 여부,{'Law': '일치하는 판례가 없습니다. 판례명을 확인하여 주십시오.'}
4,5,2024다305087,대법원,400101,민사,선고,2025.12.04,613167,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=613...,공제금등청구의소[다세대주택 건물 중 임대의뢰인 소유의 특정 세대에 대한 임대차계약 ...,{'PrecService': {'판시사항': '<br/> [1] 개업공인중개사의 중...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1712,1713,,근로복지공단산재판례,400107,일반행정,,0001.01.01,404992,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=404...,산재보험료부과처분취소,"{'PrecService': {'판시사항': '', '참조판례': '', '사건종류..."
1713,1714,,근로복지공단산재판례,400107,일반행정,,0001.01.01,382020,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=382...,유족급여및장의비부지급처분취소,"{'PrecService': {'판시사항': '', '참조판례': '', '사건종류..."
1714,1715,,근로복지공단산재판례,400107,일반행정,,0001.01.01,408336,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=408...,유족급여및장의비부지급처분취소,"{'PrecService': {'판시사항': '', '참조판례': '', '사건종류..."
1715,1716,,근로복지공단산재판례,400107,일반행정,,0001.01.01,411292,,,,/DRF/lawService.do?OC=skn24&target=prec&ID=411...,산재보험료부과처분취소,"{'PrecService': {'판시사항': '', '참조판례': '', '사건종류..."


---

### 메타데이터 / Content / 제외항목 전략 수립 및 실행

[ A. 메타데이터 ]
- 사건명
- 사건번호: ~~우리는 하나의 사건에 대한 최종 법원만 가져올 것이기 때문~~ 엄밀함을 위해서 있는 것도 좋아보임
    - 전처리할 때, 한 사건에 대해 여러 건의 판례가 있는지 확인하는 용도로 활용 가능
- 판례정보일련번호: 유지보수 관점에서 필요 (판례가 수정된 경우 업데이트해야 함)
- 선고일자: 특정 기간을 지정해서 필터링해야할 수도 있다고 판단하였음.
- 법원명: 비어있는 값은 많으나, 필터링에 사용될 가능성이 높다고 판단하였음.
- 사건종류명
- 판결유형(본: 핵심 유형 등) : 전처리 진행해보니 빈 데이터 값이 많지 않아(2건) 포함시켜도 좋을 듯 합니다.
- 참조조문: 제외 후보군1
- 참조판례: 제외 후보군2

[ B. Content ]
- 판결요지
- 판례내용
- 판시사항

[ C. 제외항목 전략 수립 ]
- 법원종류코드: 전체가 다 비었고, 법원명을 메타데이터로 올릴 것이라, 큰 메리트가 없음
- ~~판결유형: 약 1700건 중에 700건 가량이 판결유형 데이터가 존재하지 않음 + 포함해서 얻을 메리트가 크지 않아보임~~
- 선고: 억 800건 가량의 데이터가 존재하지 않음 + 포함해서 얻을 메리트가 크지 않아보임
- 판례상세링크: 주요 코드인 OC가 존재함

[ D. 아직 판단 못한 항목 ]
- 데이터출처명

---

##### 1. 기본에서 시작

In [14]:
df.info()       # 겉보기에는 멀쩡하지만 jsonl 윗부분만 보더라도 빈 문자열("")이 다수 존재함

<class 'pandas.DataFrame'>
RangeIndex: 1717 entries, 0 to 1716
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      1717 non-null   int64 
 1   사건번호    1717 non-null   str   
 2   데이터출처명  1717 non-null   str   
 3   사건종류코드  1717 non-null   int64 
 4   사건종류명   1717 non-null   str   
 5   선고      1717 non-null   str   
 6   선고일자    1717 non-null   str   
 7   판례일련번호  1717 non-null   int64 
 8   판결유형    1717 non-null   str   
 9   법원종류코드  1717 non-null   str   
 10  법원명     1717 non-null   str   
 11  판례상세링크  1717 non-null   str   
 12  사건명     1717 non-null   str   
 13  본문      1717 non-null   object
dtypes: int64(3), object(1), str(10)
memory usage: 187.9+ KB


In [15]:
# 각 컬럼별로 빈 문자열 개수
(df == "").sum()

### << Idea >>
# 1. 사건번호 같은 경우는 1700개의 비어있지 않은 문자열과 17개의 빈 문자열 값이 있음 
    # 엄밀하게 하고 싶다면 사건번호가 빈 건 온전하지 않으므로 그 판례들을 빼는 게 맞아보임
    # + 사건번호 중복 여부 파악 : 한 사건 당 하나의 (최종)판례만 존재한다면 중복이 없어야 함.
# 2. 법원명 / 판결유형 둘 모두 메타데이터로 활용하려 했으나 각각 약 40% 정도 되는 항목들이 비어 있음
    # (+) 법원명: 대법원의 판례를 보고 싶다 등의 질의는 충분히 나올 수 있는 반면에,
    # (-) 판결유형: 판결유형의 여부가 메타데이터 필터링이나 사용자 질의와 연관이 크게 없지 않을까 사료됨 (다만, 법조계 전문가가 본다면 다르게 생각할 수 있지 않을까라는 염려...)
# (-) 법원종류코드 :  다 비어있는, 무용지물
# 3. (-) 선고 : 선고도 '판결유형'과 동일한 이유로 제외하는 게 좋아보이나, 우선 어떤 데이터를 담고 있는지부터 확인해봐야 할 듯...

id           0
사건번호        17
데이터출처명       0
사건종류코드       0
사건종류명        0
선고         794
선고일자         0
판례일련번호       0
판결유형       728
법원종류코드    1717
법원명        726
판례상세링크       0
사건명          0
본문           0
dtype: int64

In [16]:
# 1. 사건 번호가 빈 케이스들 삭제
df = df[df["사건번호"] != ""]

# (df["사건번호"] == "").sum()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1700 entries, 0 to 1699
Data columns (total 14 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      1700 non-null   int64 
 1   사건번호    1700 non-null   str   
 2   데이터출처명  1700 non-null   str   
 3   사건종류코드  1700 non-null   int64 
 4   사건종류명   1700 non-null   str   
 5   선고      1700 non-null   str   
 6   선고일자    1700 non-null   str   
 7   판례일련번호  1700 non-null   int64 
 8   판결유형    1700 non-null   str   
 9   법원종류코드  1700 non-null   str   
 10  법원명     1700 non-null   str   
 11  판례상세링크  1700 non-null   str   
 12  사건명     1700 non-null   str   
 13  본문      1700 non-null   object
dtypes: int64(3), object(1), str(10)
memory usage: 186.1+ KB


In [17]:
# 1. 중복된 사건 수는 없는가?
df["사건번호"].nunique()    # 중복 사건도 있네..

1687

In [18]:
# 1. 중복 사건 조회

# (1) 중복 개수
# df["사건번호"].duplicated(keep=False).sum() # 25개의 사건이 중복됨

# (2) 중복 형태 보기
# df[df["사건번호"].duplicated(keep=False)]
    # 중복되는 모든 케이스가 사건번호가 같고, 법원명도 같은데, 판례일련번호만 다름... 본문을 대조해봐야 정확한 판단이 가능할 성 싶다.

# (3) 중복되는 모든 사건 제거
df = df[~df["사건번호"].duplicated(keep=False)]

# 여담) 중복되는 것 중 하나만 남겨도 되긴 할 듯. 혹여나 하나는 남겨도 괜쵆다는 의견이 조성되면 (3)대신 다음 코드를 이용
# df = df.drop_duplicates(subset="사건번호", keep='first')

In [19]:
df["사건번호"].duplicated(keep=False).sum()

np.int64(0)

In [20]:
# 2. '판결유형' 값 종류 확인
df["판결유형"].unique()     # 다양하긴 한데.. 해당 값이 법조계를 잘 아는 사람 아니면 큰 의미가 있는 컬럼은 아닐 것이라 생각됨.

<StringArray>
[             '판결',         '판결 : 확정',                '',          '처분청 승소',
              '결정',          '처분청 패소',              '기각',       '처분청 일부 패소',
         '판결 : 항소',         '판결 : 상고',        '전원합의체 판결',       '처분청 일부 승소',
           '처분청패소',          '판결(주1)',           '처분청승소',             '원고패',
            '추가판결',        '결정 : 재항고',          '판결: 확정',          '판결: 항소',
             '원고승',          '판결: 상고',           '판결：상고',           '판결：확정',
           '판결：항소',       '판결 : 상고기각',         '판결：상고기각',           '원고일부승',
    '판결 : 항소기각·확정',         '판결 ： 확정',     '판결：항소심 조정성립',    '판결 : 항소기각·상고',
       '판결 ： 상고기각',         '판결 ： 항소',   '판결 ： 항소기각, 확정',         '판결 ： 상고',
    '제6민사부판결 : 확정',      '민사부판결 : 항소',    '제7민사부판결 : 확정',    '제8특별부판결 : 확정',
 '제11민사부판결 : 상고기각',    '제3민사부판결 : 확정',    '제3민사부판결 : 상고',    '제2민사부판결 : 확정',
    '제4민사부판결 : 항소',   '제11민사부판결 : 확정',      '제1부판결 : 확정',    '제1민사부판결 : 확정',
      '가사부심판 : 항소',  '제1민사부판결 : 상고기각',      '특별부판결 : 확정',     

In [21]:
# 3. '선고' 값 종류 확인
df["선고"].unique()

<StringArray>
['선고', '', '자', '선고,']
Length: 4, dtype: str

---

2. jsonl 내에서 다음과 같이 표기된 케이스들 제거: "본문": {"Law": "일치하는 판례가 없습니다.  판례명을 확인하여 주십시오."}

- 결론적으로는 이 과정을 제일 먼저 수행했으면 1. 과정이 많이 생략됐을 것 같음...

In [22]:
df["본문"].astype(str).str.contains("일치하는 판례가 없습니다").sum()   # 710개나 되네요

np.int64(707)

In [23]:
df = df[~df["본문"].astype(str).str.contains("일치하는 판례가 없습니다")]

df

,id,사건번호,데이터출처명,사건종류코드,사건종류명,선고,선고일자,판례일련번호,판결유형,법원종류코드,법원명,판례상세링크,사건명,본문
0,1,2025두34604,대법원,400108,세무,선고,2026.01.29,616581,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=616...,증여세부과처분취소,{'PrecService': {'판시사항': '<br/> [1] 구 상속세 및 증여...
1,2,2025가단107133,대법원,400101,민사,선고,2026.01.23,616251,판결 : 확정,,서울북부지방법원,/DRF/lawService.do?OC=skn24&target=prec&ID=616...,배당이의,{'PrecService': {'판시사항': '<br/> 甲 소유의 주택에 관하여 ...
2,3,2025다213466,대법원,400101,민사,선고,2026.01.08,615767,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=615...,보증금,{'PrecService': {'판시사항': '<br/> [1] 주택임대차보호법 제...
4,5,2024다305087,대법원,400101,민사,선고,2025.12.04,613167,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=613...,공제금등청구의소[다세대주택 건물 중 임대의뢰인 소유의 특정 세대에 대한 임대차계약 ...,{'PrecService': {'판시사항': '<br/> [1] 개업공인중개사의 중...
8,9,2024다284418,대법원,400101,민사,선고,2025.09.11,609461,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=609...,건물인도[공공임대주택의 임대인이 임차인의 다른 주택에 관한 분양권 취득을 이유로 임...,{'PrecService': {'판시사항': '<br/> [1] 공공주택 특별법의 ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1695,1696,4287행상82,대법원,400107,일반행정,선고,1954.03.11,86050,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=860...,행정처분취소,{'PrecService': {'판시사항': ' 적법한 자격없는 자의 불하신청과 귀...
1696,1697,4287민상216,대법원,400101,민사,선고,1954.03.10,86018,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=860...,부동산소유권이전등기,{'PrecService': {'판시사항': ' 양도담보 및 대물변제 예약의 효력<...
1697,1698,4287행3,대법원,400107,일반행정,선고,1954.02.27,232305,제2특별부판결: 확정,,서울고등법원,/DRF/lawService.do?OC=skn24&target=prec&ID=232...,행정처분취소청구사건,{'PrecService': {'판시사항': ' 귀속재산처리법상의 권리가 상속될 수...
1698,1699,4287행상76,대법원,400107,일반행정,선고,1954.02.04,86048,판결,,대법원,/DRF/lawService.do?OC=skn24&target=prec&ID=860...,행정처분취소,{'PrecService': {'판시사항': ' 행정처분 전의 소원 또는 진정서의 ...


In [16]:
(df == "").sum()    # 이렇게 되면 판결 유형 정도는 다시 넣어도 되곘습니다.

id          0
사건번호        0
데이터출처명      0
사건종류코드      0
사건종류명       0
선고         67
선고일자        0
판례일련번호      0
판결유형        2
법원종류코드    968
법원명         0
판례상세링크      0
사건명         0
본문          0
dtype: int64

---

##### 3. metadata / content 분리

In [ ]:
# import json

# records = []

# for _, row in df.iterrows():
#     prec = row['본문']['PrecService']
    
#     record = {
#         "metadata": {
#             "사건명": row['사건명'],
#             "사건종류명": row['사건종류명'],
#             "판례일련번호": row['판례일련번호'],
#             "법원명": row['법원명'],
#             "선고일자": row['선고일자'],
#             "판결유형": row['판결유형'],
#             "참조조문": prec.get('참조조문', ''),
#             "참조판례": prec.get('참조판례', ''),
#             "판례상세링크": row['판례상세링크'],
#         },
#         "content": {
#             "판시사항": prec.get('판시사항', ''),
#             "판결요지": prec.get('판결요지', ''),
#             "판례내용": prec.get('판례내용', ''),
#         }
#     }
#     records.append(record)

# # (임시) JSONL로 저장
# with open('../data/processed/prec_processed.jsonl', 'w', encoding='utf-8') as f:
#     for r in records:
#         f.write(json.dumps(r, ensure_ascii=False) + '\n')

---

##### 4. 텍스트 전처리

In [24]:
import re
import json
import os
from datetime import date


# 전반적인 문자열 전처리
def clean_text(text: str) -> str:
    if not text:
        return ''
    text = re.sub(r'<br\s*/?>', '\n', text)  # <br/>은 개행
    text = re.sub(r'<[^>]+>', '', text)      # 추가적인 HTML 태그 제거
    # text = re.sub(r'\n{3,}', '\n\n', text)   # 3줄 이상 개행은 2줄로
    text = re.sub(r'[ \t]+', ' ', text)      # 연속 공백은 1칸으로
    return text.strip()


# 판례 내용에서만 있는 전처리
def clean_prec(text: str) -> str:
    if not text:
        return ''
    # 【이    유】 이후만 추출
    match = re.search(r'【이\s*유】', text)
    if match:
        text = text[match.end():]
    return clean_text(text)


# 날짜 타입 변경
def date_type(raw_date):
    y, m, d = raw_date.split(".")
    sentence_date = date(int(y), int(m), int(d)).isoformat()
    return sentence_date


# Content text 하나로 통합 (embedding을 위해)
def build_embed_text(record: dict) -> str:
    content = record["content"]
    parts = []
    
    for section_name, text in content.items():  # texts → text (str)
        if text:                                 # 빈 문자열 섹션 스킵
            parts.append(f"{section_name}: {text}")
    
    return "\n\n".join(parts)


# 함수를 활용하여 전처리
records = []

for _, row in df.iterrows():
    prec = row['본문']['PrecService']
    
    record = {
        "metadata": {
            "사건명": row['사건명'],
            "사건번호": row['사건번호'],
            "사건종류명": row['사건종류명'],
            "판례일련번호": row['판례일련번호'],
            "법원명": row['법원명'],
            "선고일자": date_type(row['선고일자']),
            "판결유형": row['판결유형'],
            # "참조조문": clean_text(prec.get('참조조문', '')),
            # "참조판례": clean_text(prec.get('참조판례', '')),
        },
        "content": {
            # "판시사항": clean_text(prec.get('판시사항', '')),
            "판결요지": clean_text(prec.get('판결요지', '')),
            "판례내용": clean_prec(prec.get('판례내용', '')),
        }
    }
    records.append(record)



path = '../data/processed/prec_processed.jsonl'

os.makedirs(os.path.dirname(path), exist_ok=True)

# 수정 전 버전 (백업)
# with open(path, 'w', encoding='utf-8') as f:
#     for r in records:
#         f.write(json.dumps(r, ensure_ascii=False) + '\n')

with open(path, 'w', encoding='utf-8') as f:
    for r in records:
        r["embed_text"] = build_embed_text(r)   # embed_text 필드 추가
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

---

---

---

# Chunking

[ Chunking ]
- 문장 단위
    - CharacterTextSplitter
    - RecursiveCharacterTextSplitter
    - KonlpyTextSplitter
- ~~SemanticChunker(유료)~~

[ Embedding Model ]
- woong0322/ko-legal-sbert-finetuned
    - 최대 시퀀스 길이: 512 토큰
    - 임베딩 차원: 768
- nlpai-lab/KURE-v1 외 0.6B 파라미터인 대부분의 모델들
    - 최대 시퀀스 길이: 8192
    - 임베딩 차원: 1024

In [3]:
# !pip install langchain-text-splitters
### requirements.txt에 추가해야함.

In [ ]:
# !pip install langchain-text-splitters konlpy
### requirements.txt에 추가해야할지 고민이 됨. >>> 안 쓸거니까 추가하지 맙시다.

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
    # 텍스트를 특정 구분자(Separator) 기준으로 단순하게 한 번만 쪼개는 도구
        # chunk_size    : 각 청크의 최대 "글자" 수 (기본값: 1000) >>> 토큰 수가 아님...
        # chunk_overlap : 앞뒤 청크가 서로 겹치는 글자 수 → 문맥 단절 방지 (기본값: 200)
        # separator     : 분할 기준 문자열 (기본값: "\n\n")
        # 구분자로 나눈 후 chunk_size를 초과하면 그냥 잘라냄 → 문맥이 끊길 수 있음
        # RecursiveCharacterTextSplitter와 달리 재시도 없이 단일 구분자로만 분할

from langchain_text_splitters import RecursiveCharacterTextSplitter
    # 문맥의 흐름을 최대한 보존하기 위해 여러 단계를 거쳐 재귀적으로 쪼개는 도구
    # 이 분할기는 미리 정의된 구분자(Separator) 목록을 가지고 있으며, 텍스트가 설정된 chunk_size보다 작아질 때까지 다음 순서에 따라 분할을 시도
        # 1단계 (\n\n): 먼저 문단 단위로 나눔
        # 2단계 (\n): 문단이 너무 크면 문장 단위(줄바꿈)로 나눔
        # 3단계 ( ): 문장이 너무 크면 단어 단위(공백)로 나눔
        # 4단계 (``): 그래도 크면 글자 단위로 나눔

from langchain_text_splitters import KonlpyTextSplitter
    # 내부적으로 형태소 분석기를 통해 텍스트를 형태소 단위로 파악한 뒤,
    # 형태소 경계에 맞춰 자연스럽게 분할 → 한국어 문맥 보존에 유리
    # 단순 글자 기준(CharacterTextSplitter)보다 의미 단위 분할이 가능

c:\Users\jinsa\anaconda3\envs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
### chunk 방식마다 chunking하는 함수 생성

import json

def chunk_character(text: str) -> list[str]:
    splitter = CharacterTextSplitter(
        chunk_size = 330,   # TextSplitter는 글자수고 임베딩 모델은 토큰수라서 그것을 고려해서 정하긴 해야함..
                            # 한글이 대략 1.5배의 토큰을 더 쓴다고 가정하면, 330글자의 한글 텍스트는 약 500개의 토큰을 사용함
        chunk_overlap = 0,  # 문맥을 유지하기 위해서 앞뒤로 중첩될 수 있는 글자열의 길이를 설정
        separator = '\n'
    )
    docs = splitter.split_text(text)
    return docs



def chunk_recursive_character(text: str) -> list:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size = 500,
        chunk_overlap = 50
    )
    docs = splitter.split_text(text)
    return docs



def chunk_konlpytextsplitter(text: str) -> list:
    splitter = KonlpyTextSplitter(
        chunk_size = 330,
        chunk_overlap = 50,
        separator = "\n"
    )
    docs = splitter.split_text(text)
    return docs

In [27]:
# chunk 진행 함수 설정
import os
import json

def chunk_prec(chunk_fn):
    output_path = f"../data/processed/prec_{chunk_fn.__name__}.jsonl"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open("../data/processed/prec_processed.jsonl", "r", encoding="utf-8") as f, open(output_path, "w", encoding="utf-8") as out:

        for line in f:
            record = json.loads(line)
            embed_text = record.get("embed_text", "").strip()

            chunks = chunk_fn(embed_text)

            for i, chunk in enumerate(chunks):
                chunk_record = {
                    "metadata": record["metadata"],   # 모든 청크에 동일한 메타데이터
                    "chunk_index": i,
                    "embed_text": chunk,              # 실제 embedding할 텍스트
                }
                out.write(json.dumps(chunk_record, ensure_ascii=False) + "\n")

In [28]:
### chunk 실제 진행

# [ 안 쓰는 chunk ]
# chunk_prec(chunk_character) # CharacterTextSplitter는 자꾸 chunk size를 초과한 것(512를 리미트로 줬는데 3300짜리도 만들고 그랬음)을 만들어서 성능이 그리 좋아보이지는 않음.
# chunk_prec(chunk_konlpytextsplitter)    # 시간이 다른 경우에 비해 상당히 많이 걸림 (좋은 컴퓨터는 아니지만, 40분 이상을 넘겨서 중단했음)>>> 시간 측면으로 그리 좋은 방식은 아닌 듯함...)
                                        # size limit도 많이 넘어가는 경우가 대다수임.

# [ 가능성이 있는 chunk ]
chunk_prec(chunk_recursive_character)   # recursive 가장 성능이 유연하고 무난한 것 같다.


In [29]:
### chunk 결과 확인

n = 146  # 원하는 번호로 변경

with open("../data/processed/prec_chunk_recursive_character.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == n:
            print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))
            break

{
  "metadata": {
    "사건명": "건물인도·소유권이전등기",
    "사건번호": "2023다264530",
    "사건종류명": "민사",
    "판례일련번호": 614307,
    "법원명": "대법원",
    "선고일자": "2025-07-18",
    "판결유형": "판결"
  },
  "chunk_index": 16,
  "embed_text": "4. 결론\n 나머지 상고이유에 관한 판단을 생략한 채 원심판결을 파기하고, 사건을 다시 심리·판단하도록 원심법원에 환송하기로 하여, 관여 대법관의 일치된 의견으로 주문과 같이 판결한다."
}


### Chunking 성능 확인 >>> VectorDB 구축해보면서 해볼까?

---

---

---

# VectorDB (QdrantStore) 구축 및 테스트

In [4]:
# !pip install sentence-transformers qdrant-client
### 이 둘 모듈도 추가해야함

In [8]:
# !pip install -U sentence-transformers transformers torch

In [ ]:
# !pip install openai
### 이 모듈 추가해야함

In [1]:
import sys
sys.path.insert(0, "..")

import json
from src.vectordb import Embedder, QdrantStore
from tqdm import tqdm



embedder = Embedder()

print(embedder._model)
print(embedder.vector_size)

store = QdrantStore("prec_text-embedding-3-small_3", embedder)

texts, metadatas = [], []

BATCH_SIZE = 256
with open("../data/processed/prec_chunk_recursive_character.jsonl", encoding="utf-8") as f:
    lines = f.readlines()

for line in tqdm(lines, desc="임베딩 중"):
    obj = json.loads(line)
    texts.append(obj["embed_text"])
    metadatas.append(obj["metadata"])

    if len(texts) >= BATCH_SIZE:
        store.add_docs(texts=texts, metadatas=metadatas)
        texts, metadatas = [], []

if texts:
    store.add_docs(texts=texts, metadatas=metadatas)

# 이전 버전 코드
# with open("../data/processed/prec_chunk_recursive_character.jsonl", encoding="utf-8") as f:
#     data = [json.loads(line) for line in f]

# texts = [obj["embed_text"] for obj in data]
# metadatas = [obj["metadata"] for obj in data]

# store.add_docs(texts=texts, metadatas=metadatas)

print("완료!")

text-embedding-3-small
1536


c:\Users\jinsa\OneDrive\SKN24\project_files\3rd_project\SKN24_3rd_6team\notebooks\..\src\vectordb\store.py:29: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <prec_text-embedding-3-small> contains 46537 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self._client = QdrantClient(path=path)
c:\Users\jinsa\OneDrive\SKN24\project_files\3rd_project\SKN24_3rd_6team\notebooks\..\src\vectordb\store.py:29: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <prec_text-embedding-3-small_2> contains 30488 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  self._client = QdrantClient(path=path)
임베딩 중:  66%|██████▋   | 19968/30035 [04:47<02:20, 71.81it/s]c:\Users\jinsa\OneDrive\SKN24\project_files\3rd_project\SKN24_3rd_6team\notebooks\..\src\vectordb\store.py:74: UserWarning: Local mode is not re

완료!


In [2]:
count = store._client.get_collection("prec_text-embedding-3-small_3").points_count
print(f"저장된 벡터 수: {count}")
print(f"총 라인 수: {len(lines)}")

저장된 벡터 수: 30035
총 라인 수: 30035


In [3]:
print(store._collection)  # 컬렉션 이름 확인

prec_text-embedding-3-small_3


In [ ]:
### Query 예시

# 판례 같은 경우에는

# 위약금 면책 조항이 있는데 임대차 계약 중도 퇴실 위약금을 부담하래요

In [9]:
results = store.search("임대차 계약 중도 해지 특약", top_k=5)
for r in results:
    print(f"score: {r['score']:.4f}")
    print(f"text: {r['text']}")

    metadata = {k: v for k, v in r.items() if k not in ("score", "text")}
    print(metadata)  # r 자체에 메타데이터 포함
    
    print()

score: 0.6104
text: 임대차계약을 체결할 것을 의욕하였으리라고 볼 수 있는지 명확하지 않은데다가, 위 상한액의 초과 부분이 정기예금이율에 기초한 전환율에 따라 임대료로 전환된다고 단정하거나 그와 같은 법적 효과가 발생한다고 인정할 만한 근거도 부족하다.
{'사건명': '부당이득금등', '사건번호': '2018나2043218', '사건종류명': '민사', '판례일련번호': 222839, '법원명': '서울고등법원', '선고일자': '2020-07-02', '판결유형': '판결'}

score: 0.5904
text: 임대인은 그 임대차계약을 해지하거나 임대차계약의 갱신을 거절할 수 있다(대법원 2005. 4. 29. 선고 2005다8002 판결 등 참조).
{'사건명': '건물명도·손해배상(기)', '사건번호': '2016다241805', '사건종류명': '민사', '판례일련번호': 193378, '법원명': '대법원', '선고일자': '2018-02-08', '판결유형': '판결'}

score: 0.5412
text: 임대차계약 자체가 무효로 된다거나 또는 그 임대차계약에서 정한 차임이 무효가 되는 임대보증금 부분에 상응하여 그만큼 증액된다고 보아야 할 아무런 이유가 없다.
{'사건명': '건물인도등(표준임대료와 당초 계약상 임대료의 차액 연체를 이유로 임대차계약을 해지하고 임대주택의 인도를 구하는 사건)', '사건번호': '2013다42236', '사건종류명': '민사', '판례일련번호': 187884, '법원명': '대법원', '선고일자': '2016-11-18', '판결유형': '전원합의체 판결'}

score: 0.5369
text: 임대인이 임차보증금을 반환할 사유가 발생하였음에도 계약자가 임차보증금을 반환받지 못한 때제3조(보상하지 않는 손해)② 회사는 제20조(계약의 변경 및 갱신)에 규정한 사유가 발생하였음에도 이 계약을 갱신 또는 변경하지 않은 상태에서 발생한 손해는 보상하지 않습니다.제6조(보험금의 청구)② 회사는 제1항에 따

---

---

---

### Cf. 개선점
- 전처리 단계에서 더욱 세밀하게 조정할 수 있지 않았을지.
    - 가령, 1./2./3. 넘버링이나 가./나./다. 로 순서를 정한 것들에 대한 전처리가 가능했는지 여부